# Brent 실험 전체 재현 노트북

이 노트북은 이 대화에서 진행한 **df.csv 기반 surrogate 실험 전체**를 재현하도록 구성되어 있습니다.

재현 범위:
- `brent_experiment_run` 1차 실험
- `brent_experiment_round2` 2차 실험
- `compliant_experiment_round` 준수형(strict) 실험

실행 모드:
- **snapshot replay**: 이전에 생성해 둔 결과 스냅샷이 있으면 그대로 복원합니다. 이 모드가 가장 빠르고, 대화에서 보고한 표/CSV/리포트를 그대로 다시 만듭니다.
- **recompute**: `df.csv`만으로 surrogate 실험을 다시 학습/평가합니다. 시드와 로직은 고정했지만, 환경 차이로 숫자는 약간 달라질 수 있습니다.

제약:
- repo `main.py` 실험이 아니라, 이 대화에서 제가 실제로 돌렸던 **df-only surrogate 실험 계열**을 코드화한 것입니다.
- `Com_Oil_Spread`는 업로드된 `df.csv`에 없으면 자동으로 제외됩니다.


In [ ]:

from pathlib import Path
import os, math, random, shutil, zipfile, json, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
torch.set_num_threads(1)

# ---------- user-facing toggles ----------
MODE = "snapshot_if_available"   # "snapshot_if_available" | "snapshot_only" | "recompute"
FORCE_RECOMPUTE = False          # True면 snapshot이 있어도 다시 계산
SAVE_EXECUTION_BUNDLE = True

TARGET = "Com_BrentCrudeOil"
ALLOWED = [
    "GPRD_THREAT", "BS_Core_Index_A", "GPRD", "GPRD_ACT", "BS_Core_Index_B",
    "BS_Core_Index_C", "Idx_OVX", "Com_Oil_Spread", "Com_LMEX", "Com_BloombergCommodity_BCOM"
]
STAR = ["GPRD_THREAT", "BS_Core_Index_A", "BS_Core_Index_C", "Idx_OVX"]

H = 2
TRAIN_END_MAX = 554
VAL_END_IDX = list(range(555, 579))
TEST_END_IDX = [580]

def find_df_csv():
    candidates = [
        Path.cwd() / "df.csv",
        Path("/mnt/data/df.csv"),
        Path.cwd().parent / "df.csv",
    ]
    env_path = os.environ.get("DF_PATH")
    if env_path:
        candidates.insert(0, Path(env_path))
    for p in candidates:
        if p.exists():
            return p.resolve()
    raise FileNotFoundError("df.csv를 찾지 못했습니다. notebook과 같은 폴더나 /mnt/data 에 df.csv를 두세요.")

NOTEBOOK_DIR = Path.cwd()
DF_PATH = find_df_csv()
SNAPSHOT_ROOT_CANDIDATES = [
    NOTEBOOK_DIR / "brent_full_experiment_snapshots",
    Path("/mnt/data/brent_full_experiment_snapshots"),
]
SNAPSHOT_ROOT = next((p for p in SNAPSHOT_ROOT_CANDIDATES if p.exists()), None)

OUT_RUN = NOTEBOOK_DIR / "brent_experiment_run"
OUT_ROUND2 = NOTEBOOK_DIR / "brent_experiment_round2"
OUT_COMPLIANT = NOTEBOOK_DIR / "compliant_experiment_round"

print("DF_PATH:", DF_PATH)
print("SNAPSHOT_ROOT:", SNAPSHOT_ROOT)
print("NOTEBOOK_DIR:", NOTEBOOK_DIR.resolve())


In [ ]:

def copy_tree(src: Path, dst: Path):
    if dst.exists():
        # keep existing output folder if already present; avoids permission issues in shared runtimes
        return "kept_existing"
    shutil.copytree(src, dst)
    return "copied"

def replay_snapshots(snapshot_root: Path):
    copied = []
    for name in ["brent_experiment_run", "brent_experiment_round2", "compliant_experiment_round"]:
        src = snapshot_root / name
        dst = NOTEBOOK_DIR / name
        if src.exists():
            action = copy_tree(src, dst)
            copied.append((str(dst), action))
    return copied

replayed = []
if MODE in {"snapshot_if_available", "snapshot_only"} and SNAPSHOT_ROOT is not None and not FORCE_RECOMPUTE:
    replayed = replay_snapshots(SNAPSHOT_ROOT)
    print("Snapshot replay status:")
    for p, action in replayed:
        print(f" - {p} ({action})")
elif MODE == "snapshot_only":
    raise RuntimeError("snapshot_only 모드인데 snapshot folder를 찾지 못했습니다.")
else:
    print("No snapshot replay; recompute path will run.")


In [ ]:

df_raw = pd.read_csv(DF_PATH)
if "dt" not in df_raw.columns:
    raise ValueError("df.csv must include a 'dt' column.")
df_raw["dt"] = pd.to_datetime(df_raw["dt"])

present = [c for c in ALLOWED if c in df_raw.columns]
missing = [c for c in ALLOWED if c not in df_raw.columns]
if TARGET not in df_raw.columns:
    raise ValueError(f"{TARGET} not found in df.csv")

data = df_raw[["dt", TARGET] + present].copy()
for c in [TARGET] + present:
    data[f"diff_{c}"] = data[c].diff()
data = data.iloc[1:].reset_index(drop=True)

actual_holdout = data.loc[TEST_END_IDX[0] + 1 : TEST_END_IDX[0] + H, ["dt", TARGET]].copy()

print("present exog:", present)
print("missing exog:", missing)
print("transformed rows:", len(data))
print("holdout actual:")
display(actual_holdout)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def pred_levels(anchor, pred_diff):
    return anchor[:, None] + torch.cumsum(pred_diff, dim=1)

def mape_np(y_true, y_pred):
    return float(np.mean(np.abs((y_true - y_pred) / np.clip(np.abs(y_true), 1e-8, None))))

def nrmse_np(y_true, y_pred):
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    denom = float(np.std(y_true))
    return rmse / (denom + 1e-8)

def compute_feature_frame(train_end_idx, use_aa=True, neg_weight=1.0):
    base_cols = [f"diff_{TARGET}"] + [f"diff_{c}" for c in present]
    scaler = StandardScaler().fit(data.loc[:train_end_idx, base_cols])
    feat = pd.DataFrame(
        scaler.transform(data[base_cols]),
        columns=base_cols,
        index=data.index,
    )
    stats = {"scaler_mean": scaler.mean_.tolist(), "scaler_scale": scaler.scale_.tolist()}
    if not use_aa:
        return feat, stats, base_cols

    star_present = [c for c in STAR if c in present]
    critical_ratio = np.zeros(len(data), dtype=np.float32)
    robust = {}
    for c in star_present:
        series = data.loc[:train_end_idx, f"diff_{c}"].values
        med = float(np.median(series))
        mad = float(np.median(np.abs(series - med)))
        if mad < 1e-6:
            mad = float(np.std(series) / 1.4826 + 1e-6)
        z = (data[f"diff_{c}"].values - med) / (1.4826 * mad + 1e-6)
        pos = np.maximum(z, 0.0).astype(np.float32)
        neg = (np.maximum(-z, 0.0) * neg_weight).astype(np.float32)
        crit = (np.abs(z) > 1.5).astype(np.float32)

        critical_ratio += crit
        feat[f"aa_pos_{c}"] = pos
        feat[f"aa_neg_{c}"] = neg
        robust[c] = {"median": med, "mad": mad}

    feat["aa_critical_ratio"] = critical_ratio / max(1, len(star_present))
    stats["robust"] = robust
    return feat, stats, base_cols

def build_event_map(feat_df, input_size):
    aa_cols = [c for c in feat_df.columns if c.startswith("aa_")]
    pos_cols = [c for c in aa_cols if c.startswith("aa_pos_")]
    event_score_by_end = {}
    event_profile_by_end = {}
    for end in range(input_size - 1, len(feat_df) - H):
        window = feat_df.loc[end - input_size + 1 : end]
        if pos_cols:
            pos_tail = window[pos_cols].tail(8).to_numpy().sum()
            crit_tail = window["aa_critical_ratio"].tail(8).sum() if "aa_critical_ratio" in window else 0.0
            score = float(pos_tail + 2.0 * crit_tail)
            profile = np.array(
                [window[c].tail(8).mean() for c in pos_cols] + [float(crit_tail / 8.0)],
                dtype=np.float32,
            )
        else:
            score = 0.0
            profile = np.zeros(1, dtype=np.float32)
        event_score_by_end[end] = score
        event_profile_by_end[end] = profile
    return event_score_by_end, event_profile_by_end

def make_windows(feat_df, end_indices, input_size):
    X, y_diff, y_level, anchor = [], [], [], []
    dates = []

    feat_arr = feat_df.values.astype(np.float32)
    target_diff = data[f"diff_{TARGET}"].values.astype(np.float32)
    target_level = data[TARGET].values.astype(np.float32)

    for t in end_indices:
        X.append(feat_arr[t - input_size + 1 : t + 1])
        y_diff.append(target_diff[t + 1 : t + H + 1])
        y_level.append(target_level[t + 1 : t + H + 1])
        anchor.append(target_level[t])
        dates.append(
            (
                str(data.loc[t, "dt"].date()),
                str(data.loc[t + 1, "dt"].date()),
                str(data.loc[t + 2, "dt"].date()),
            )
        )

    return {
        "X": torch.tensor(np.stack(X), dtype=torch.float32),
        "y_diff": torch.tensor(np.stack(y_diff), dtype=torch.float32),
        "y_level": torch.tensor(np.stack(y_level), dtype=torch.float32),
        "anchor": torch.tensor(np.array(anchor), dtype=torch.float32),
        "end_indices": np.array(end_indices, dtype=int),
        "dates": dates,
    }

def compute_sample_weights(end_indices, event_score_by_end, scheme="uniform", strength=0.0, threshold_q=0.9):
    scores = np.array([event_score_by_end[int(e)] for e in end_indices], dtype=np.float32)
    if len(scores) == 0:
        return np.ones(0, dtype=np.float32), 0.0
    thr = float(np.quantile(scores, threshold_q))
    if scheme == "uniform" or strength <= 0:
        w = np.ones_like(scores)
    elif scheme == "binary":
        w = 1.0 + strength * (scores >= thr).astype(np.float32)
    elif scheme == "soft":
        scale = max(scores.std(), 1e-6)
        uplift = np.maximum(scores - thr, 0.0) / scale
        w = 1.0 + strength * uplift
    else:
        raise ValueError(scheme)
    return w.astype(np.float32), thr


In [ ]:

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, : x.size(1)]

class PlainGRU(nn.Module):
    def __init__(self, input_dim, hidden=16):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden * 2, hidden),
            nn.ReLU(),
            nn.Linear(hidden, H),
        )

    def encode(self, x):
        out, _ = self.gru(x)
        summary = torch.cat([out[:, -1], out[:, -8:].mean(1)], dim=-1)
        return summary

    def decode_summary(self, summary):
        return self.head(summary)

    def forward(self, x, return_summary=False):
        summary = self.encode(x)
        pred = self.decode_summary(summary)
        return (pred, summary) if return_summary else pred

class PlainTransformer(nn.Module):
    def __init__(self, input_dim, d_model=24, nhead=4, num_layers=1, dropout=0.05):
        super().__init__()
        self.proj = nn.Linear(input_dim, d_model)
        self.pos = PositionalEncoding(d_model, max_len=512)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 2,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.head = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Linear(d_model, H),
        )

    def encode(self, x):
        h = self.encoder(self.pos(self.proj(x)))
        summary = torch.cat([h[:, -1], h[:, -8:].mean(1)], dim=-1)
        return summary

    def decode_summary(self, summary):
        return self.head(summary)

    def forward(self, x, return_summary=False):
        summary = self.encode(x)
        pred = self.decode_summary(summary)
        return (pred, summary) if return_summary else pred

class AAGRU(nn.Module):
    def __init__(self, input_dim, base_dim, hidden=20):
        super().__init__()
        self.base_dim = base_dim
        self.gru = nn.GRU(input_dim, hidden, batch_first=True)
        star_dim = max(1, input_dim - base_dim)
        self.star_gate = nn.Sequential(
            nn.Linear(star_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.Sigmoid(),
        )
        self.head = nn.Sequential(
            nn.Linear(hidden * 3, hidden),
            nn.ReLU(),
            nn.Linear(hidden, H),
        )

    def encode(self, x):
        out, _ = self.gru(x)
        last = out[:, -1]
        pooled = out[:, -8:].mean(1)
        star_last = x[:, -1, self.base_dim :]
        gate = self.star_gate(star_last)
        summary = torch.cat([last, pooled, last * gate], dim=-1)
        return summary

    def decode_summary(self, summary):
        return self.head(summary)

    def forward(self, x, return_summary=False):
        summary = self.encode(x)
        pred = self.decode_summary(summary)
        return (pred, summary) if return_summary else pred

class AATransformerJoint(nn.Module):
    def __init__(self, input_dim, base_dim, d_model=24, nhead=4, num_layers=1, dropout=0.05):
        super().__init__()
        self.base_dim = base_dim
        self.proj = nn.Linear(input_dim, d_model)
        self.pos = PositionalEncoding(d_model, max_len=512)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 2,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.query = nn.Parameter(torch.randn(H, d_model) * 0.02)
        self.q_self = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.cross = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        star_dim = max(1, input_dim - base_dim)
        self.star_context = nn.Sequential(
            nn.Linear(star_dim, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
        self.token_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, 1),
        )

    def encode_tokens(self, x):
        return self.encoder(self.pos(self.proj(x)))

    def encode(self, x):
        h = self.encode_tokens(x)
        return torch.cat([h[:, -1], h[:, -8:].mean(1)], dim=-1)

    def decode_from_tokens(self, h, x):
        b = x.size(0)
        q = self.query.unsqueeze(0).expand(b, -1, -1)
        star_summary = x[:, -8:, self.base_dim :].mean(1)
        q = q + self.star_context(star_summary).unsqueeze(1)
        q, _ = self.q_self(q, q, q, need_weights=False)
        q, _ = self.cross(q, h, h, need_weights=False)
        return self.token_head(q).squeeze(-1)

    def forward(self, x, return_summary=False):
        h = self.encode_tokens(x)
        pred = self.decode_from_tokens(h, x)
        summary = torch.cat([h[:, -1], h[:, -8:].mean(1)], dim=-1)
        return (pred, summary) if return_summary else pred


In [ ]:

def weighted_smooth_l1(pred_level, true_level, sample_weights=None):
    loss = F.smooth_l1_loss(pred_level, true_level, reduction="none").mean(dim=1)
    if sample_weights is None:
        return loss.mean()
    w = sample_weights.to(loss.device)
    return (loss * w).sum() / w.sum().clamp_min(1e-8)

def train_model(model, train_ds, val_ds, seed=42, epochs=35, lr=2e-3, wd=1e-4, sample_weights=None):
    set_seed(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    best = None
    best_state = None
    patience = 8
    bad = 0
    sw = None if sample_weights is None else torch.tensor(sample_weights, dtype=torch.float32)

    for epoch in range(epochs):
        model.train()
        opt.zero_grad()

        pred_diff = model(train_ds["X"])
        pred_level = pred_levels(train_ds["anchor"], pred_diff)
        loss = weighted_smooth_l1(pred_level, train_ds["y_level"], sw)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        model.eval()
        with torch.no_grad():
            val_pred_diff = model(val_ds["X"])
            val_pred_level = pred_levels(val_ds["anchor"], val_pred_diff).cpu().numpy()

        vmape = mape_np(val_ds["y_level"].numpy(), val_pred_level)
        vnrmse = nrmse_np(val_ds["y_level"].numpy(), val_pred_level)
        vup = float(np.mean(val_pred_level[:, 1] > val_pred_level[:, 0]))
        score = vmape + 0.12 * vnrmse - 0.01 * vup

        if best is None or score < best["score"]:
            best = {
                "epoch": epoch,
                "score": score,
                "mape": vmape,
                "nrmse": vnrmse,
                "uptrend_rate": vup,
            }
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    model.load_state_dict(best_state)
    return model, best

def evaluate_model(model, ds):
    model.eval()
    with torch.no_grad():
        pred_diff, summary = model(ds["X"], return_summary=True)
        pred_level = pred_levels(ds["anchor"], pred_diff).cpu().numpy()
    true = ds["y_level"].cpu().numpy()
    return {
        "pred_level": pred_level,
        "pred_diff": pred_diff.cpu().numpy(),
        "summary": summary.cpu().numpy(),
        "true_level": true,
        "mape": mape_np(true, pred_level),
        "nrmse": nrmse_np(true, pred_level),
        "uptrend_rate": float(np.mean(pred_level[:, 1] > pred_level[:, 0])),
    }

def fit_base_variant(name, use_aa=False, neg_weight=1.0, input_size=64, seed=0,
                     weight_scheme="uniform", weight_strength=0.0, threshold_q=0.9):
    feat, stats, base_cols = compute_feature_frame(TRAIN_END_MAX, use_aa=use_aa, neg_weight=neg_weight)
    train_end_idx = list(range(input_size - 1, 555))
    val_end_idx = list(range(555, 579))
    test_end_idx = [580]

    train_ds = make_windows(feat, train_end_idx, input_size)
    val_ds = make_windows(feat, val_end_idx, input_size)
    test_ds = make_windows(feat, test_end_idx, input_size)

    event_map, event_profiles = build_event_map(feat, input_size)
    sample_weights, weight_thr = compute_sample_weights(
        train_end_idx, event_map, scheme=weight_scheme, strength=weight_strength, threshold_q=threshold_q
    )

    if name == "plain_gru":
        model = PlainGRU(train_ds["X"].shape[-1], hidden=16)
    elif name == "plain_transformer":
        model = PlainTransformer(train_ds["X"].shape[-1], d_model=24, nhead=4, num_layers=1, dropout=0.05)
    elif name == "aa_gru":
        model = AAGRU(train_ds["X"].shape[-1], base_dim=len(base_cols), hidden=20)
    elif name == "aa_transformer_joint":
        model = AATransformerJoint(train_ds["X"].shape[-1], base_dim=len(base_cols), d_model=24, nhead=4, num_layers=1, dropout=0.05)
    else:
        raise ValueError(name)

    model, best = train_model(
        model, train_ds, val_ds, seed=seed, epochs=35, lr=2e-3, wd=1e-4, sample_weights=sample_weights
    )
    val_eval = evaluate_model(model, val_ds)
    test_eval = evaluate_model(model, test_ds)

    row = {
        "variant": name,
        "input_size": input_size,
        "seed": seed,
        "use_aa": use_aa,
        "neg_weight": neg_weight,
        "weight_scheme": weight_scheme,
        "weight_strength": weight_strength,
        "threshold_q": threshold_q,
        "val_mape": val_eval["mape"],
        "val_nrmse": val_eval["nrmse"],
        "val_uptrend_rate": val_eval["uptrend_rate"],
        "holdout_h1_pred": float(test_eval["pred_level"][0, 0]),
        "holdout_h2_pred": float(test_eval["pred_level"][0, 1]),
        "holdout_h1_actual": float(test_eval["true_level"][0, 0]),
        "holdout_h2_actual": float(test_eval["true_level"][0, 1]),
        "holdout_h1_err_pct": abs(float(test_eval["pred_level"][0, 0] - test_eval["true_level"][0, 0])) / float(test_eval["true_level"][0, 0]),
        "holdout_h2_err_pct": abs(float(test_eval["pred_level"][0, 1] - test_eval["true_level"][0, 1])) / float(test_eval["true_level"][0, 1]),
        "holdout_uptrend": bool(test_eval["pred_level"][0, 1] > test_eval["pred_level"][0, 0]),
        "selector_epoch": best["epoch"],
        "selector_score": best["score"],
        "weight_threshold": weight_thr,
    }

    pack = {
        "row": row,
        "feat": feat,
        "stats": stats,
        "base_cols": base_cols,
        "train_ds": train_ds,
        "val_ds": val_ds,
        "test_ds": test_ds,
        "event_map": event_map,
        "event_profiles": event_profiles,
        "model": model,
        "val_eval": val_eval,
        "test_eval": test_eval,
        "input_size": input_size,
        "name": name,
    }
    return row, pack

def precompute_all_windows(pack):
    input_size = pack["input_size"]
    all_end_idx = list(range(input_size - 1, len(data) - H))
    all_ds = make_windows(pack["feat"], all_end_idx, input_size)
    all_eval = evaluate_model(pack["model"], all_ds)
    summary_by_end = {int(e): s for e, s in zip(all_ds["end_indices"], all_eval["summary"])}
    pred_by_end = {int(e): p for e, p in zip(all_ds["end_indices"], all_eval["pred_level"])}
    true_by_end = {int(e): y for e, y in zip(all_ds["end_indices"], all_eval["true_level"])}
    anchor_by_end = {int(e): float(a) for e, a in zip(all_ds["end_indices"], all_ds["anchor"].numpy())}
    return {
        "all_end_idx": all_end_idx,
        "all_ds": all_ds,
        "summary_by_end": summary_by_end,
        "pred_by_end": pred_by_end,
        "true_by_end": true_by_end,
        "anchor_by_end": anchor_by_end,
    }

def build_key(summary, event_profile=None, key_mode="summary", beta=None):
    if key_mode == "summary":
        key = np.asarray(summary, dtype=np.float32)
    elif key_mode == "summary_event":
        if event_profile is None:
            raise ValueError("event_profile required")
        b = 1.0 if beta is None else float(beta)
        key = np.concatenate([np.asarray(summary, dtype=np.float32), b * np.asarray(event_profile, dtype=np.float32)])
    else:
        raise ValueError(key_mode)
    return key / (np.linalg.norm(key) + 1e-8)

def level_from_value_mode(mem_values, anchor_current, value_mode):
    mem_values = np.asarray(mem_values, dtype=np.float32)
    if value_mode == "level":
        return mem_values
    elif value_mode == "delta":
        return anchor_current + mem_values
    elif value_mode == "return":
        return anchor_current * (1.0 + mem_values)
    else:
        raise ValueError(value_mode)

def compute_value_store(cache, value_mode="level"):
    store = {}
    for e, future in cache["true_by_end"].items():
        anchor = cache["anchor_by_end"][e]
        if value_mode == "level":
            store[e] = future.astype(np.float32)
        elif value_mode == "delta":
            store[e] = (future - anchor).astype(np.float32)
        elif value_mode == "return":
            store[e] = (future / anchor - 1.0).astype(np.float32)
        else:
            raise ValueError(value_mode)
    return store

def retrieve_prediction_for_end(pack, cache, query_end, trigger_q=0.9, ratio_floor=0.9,
                                key_mode="summary", beta=None, value_mode="level",
                                top_k=1, alpha=0.25, recency_gap=8, candidate_pool_end=TRAIN_END_MAX):
    event_map = pack["event_map"]
    event_profiles = pack["event_profiles"]
    train_scores = np.array([event_map[e] for e in range(pack["input_size"] - 1, candidate_pool_end + 1)], dtype=np.float32)
    trigger_thr = float(np.quantile(train_scores, trigger_q))
    query_score = event_map[int(query_end)]

    base_pred = cache["pred_by_end"][int(query_end)]
    base_summary = cache["summary_by_end"][int(query_end)]
    anchor_current = cache["anchor_by_end"][int(query_end)]

    if query_score < trigger_thr:
        return {
            "pred": base_pred.copy(),
            "base_pred": base_pred.copy(),
            "triggered": False,
            "trigger_threshold": trigger_thr,
            "query_score": query_score,
            "neighbors": [],
            "similarities": [],
        }

    candidates = [e for e in cache["all_end_idx"] if e <= int(query_end) - recency_gap and e <= candidate_pool_end]
    eligible = [e for e in candidates if event_map[e] >= ratio_floor * query_score]
    if not eligible:
        eligible = candidates

    qk = build_key(base_summary, event_profiles[int(query_end)], key_mode=key_mode, beta=beta)
    ref_keys = np.stack([build_key(cache["summary_by_end"][e], event_profiles[e], key_mode=key_mode, beta=beta) for e in eligible])
    sims = ref_keys @ qk
    order = np.argsort(sims)[-top_k:][::-1]
    chosen = [eligible[i] for i in order.tolist()]
    sim_vals = sims[order]

    weights = np.maximum(sim_vals, 0.0)
    if np.allclose(weights.sum(), 0.0):
        weights = np.ones_like(weights)
    weights = weights / weights.sum()

    value_store = compute_value_store(cache, value_mode=value_mode)
    mem_raw = np.sum(np.stack([value_store[e] for e in chosen]) * weights[:, None], axis=0)
    mem_level = level_from_value_mode(mem_raw, anchor_current, value_mode=value_mode)
    final = (1 - alpha) * base_pred + alpha * mem_level

    return {
        "pred": final.astype(np.float32),
        "base_pred": base_pred.copy(),
        "memory_level": mem_level.astype(np.float32),
        "triggered": True,
        "trigger_threshold": trigger_thr,
        "query_score": query_score,
        "neighbors": chosen,
        "similarities": sim_vals.tolist(),
    }

def historical_probe_rule(pack, rule, key_mode="summary", beta=None, value_mode="level", recency_gap=8):
    cache = precompute_all_windows(pack)
    rows = []
    probe_end_idx = list(range(pack["input_size"] - 1, len(data) - H))
    gains = []
    for e in probe_end_idx:
        gains.append(float(cache["true_by_end"][e][1] - cache["anchor_by_end"][e]))
    surge_thr = float(np.quantile(np.array(gains), 0.9))

    for e in probe_end_idx:
        res = retrieve_prediction_for_end(
            pack, cache, e,
            trigger_q=rule["trigger_q"],
            ratio_floor=rule["ratio_floor"],
            key_mode=key_mode,
            beta=beta,
            value_mode=value_mode,
            top_k=rule["top_k"],
            alpha=rule["alpha"],
            recency_gap=recency_gap,
        )
        future = cache["true_by_end"][e]
        anchor = cache["anchor_by_end"][e]
        base = cache["pred_by_end"][e]
        rows.append({
            "end_idx": e,
            "dt_end": str(data.loc[e, "dt"].date()),
            "event_score": pack["event_map"][e],
            "triggered": res["triggered"],
            "base_h1": float(base[0]),
            "base_h2": float(base[1]),
            "rule_h1": float(res["pred"][0]),
            "rule_h2": float(res["pred"][1]),
            "actual_h1": float(future[0]),
            "actual_h2": float(future[1]),
            "base_h2_err": abs(float(base[1] - future[1])) / float(future[1]),
            "rule_h2_err": abs(float(res["pred"][1] - future[1])) / float(future[1]),
            "gain_true_h2": float(future[1] - anchor),
            "neighbors": str(res["neighbors"]),
            "improved_h2": abs(float(res["pred"][1] - future[1])) < abs(float(base[1] - future[1])),
            "surge": float(future[1] - anchor) >= surge_thr,
        })

    probe_df = pd.DataFrame(rows)
    triggered = probe_df[probe_df["triggered"]]
    surge_triggered = triggered[triggered["surge"]]
    summary = {
        "n_total": int(len(probe_df)),
        "n_triggered": int(len(triggered)),
        "trigger_rate": float(len(triggered) / len(probe_df)),
        "triggered_improved_h2_rate": float(triggered["improved_h2"].mean()) if len(triggered) else 0.0,
        "triggered_base_h2_err_mean": float(triggered["base_h2_err"].mean()) if len(triggered) else np.nan,
        "triggered_rule_h2_err_mean": float(triggered["rule_h2_err"].mean()) if len(triggered) else np.nan,
        "surge_triggered_count": int(len(surge_triggered)),
        "surge_base_h2_err_mean": float(surge_triggered["base_h2_err"].mean()) if len(surge_triggered) else np.nan,
        "surge_rule_h2_err_mean": float(surge_triggered["rule_h2_err"].mean()) if len(surge_triggered) else np.nan,
    }
    return probe_df, summary

def choose_best_row(df):
    return df.sort_values("selector_score", ascending=True).iloc[0].to_dict()


## Round 1
기본/AA/retrieval overlay/gated retrieval/repeatability/surge probe를 재현합니다.

In [ ]:

if FORCE_RECOMPUTE or not all((NOTEBOOK_DIR / p).exists() for p in ["brent_experiment_run"]):
    print("Running round-1 experiments...")
    OUT_RUN.mkdir(exist_ok=True, parents=True)

    base_specs = [
        ("plain_gru", False, 1.0, 64, 0, "uniform", 0.0, 0.9),
        ("plain_transformer", False, 1.0, 64, 0, "uniform", 0.0, 0.9),
        ("aa_gru", True, 1.0, 64, 0, "uniform", 0.0, 0.9),
        ("aa_transformer_joint", True, 1.0, 64, 0, "uniform", 0.0, 0.9),
        ("aa_transformer_joint", True, 0.95, 64, 0, "uniform", 0.0, 0.9),
        ("aa_transformer_joint", True, 0.90, 64, 0, "uniform", 0.0, 0.9),
    ]

    round1_rows = []
    round1_packs = {}
    for spec in base_specs:
        name, use_aa, neg_weight, input_size, seed, ws, ws_strength, tq = spec
        row, pack = fit_base_variant(name, use_aa, neg_weight, input_size, seed, ws, ws_strength, tq)
        round1_rows.append(row)
        round1_packs[(name, neg_weight)] = pack

    df_results = pd.DataFrame(round1_rows).sort_values(["variant", "neg_weight"]).reset_index(drop=True)
    df_results.to_csv(OUT_RUN / "preliminary_results.csv", index=False)

    retrieval_rows = []
    for neg_weight in [1.00, 0.95, 0.90]:
        pack = round1_packs[("aa_transformer_joint", neg_weight)]
        cache = precompute_all_windows(pack)
        records = []
        for alpha in [0.25, 0.5, 0.75, 1.0]:
            preds = []
            triggered = 0
            for end in pack["val_ds"]["end_indices"]:
                res = retrieve_prediction_for_end(
                    pack, cache, int(end),
                    trigger_q=0.0,
                    ratio_floor=0.0,
                    key_mode="summary",
                    value_mode="level",
                    top_k=1,
                    alpha=alpha,
                    recency_gap=8,
                )
                preds.append(res["pred"])
                triggered += int(res["triggered"])
            preds = np.stack(preds)
            true = pack["val_eval"]["true_level"]
            score = mape_np(true, preds) + 0.12 * nrmse_np(true, preds) - 0.01 * float(np.mean(preds[:, 1] > preds[:, 0]))
            records.append({
                "alpha": alpha,
                "val_mape": mape_np(true, preds),
                "val_nrmse": nrmse_np(true, preds),
                "val_uptrend_rate": float(np.mean(preds[:, 1] > preds[:, 0])),
                "selector_score": score,
                "triggered_val_count": triggered,
            })
        score_df = pd.DataFrame(records).sort_values("selector_score").reset_index(drop=True)
        best_alpha = float(score_df.iloc[0]["alpha"])
        holdout = retrieve_prediction_for_end(
            pack, cache, 580, trigger_q=0.0, ratio_floor=0.0,
            key_mode="summary", value_mode="level", top_k=1, alpha=best_alpha, recency_gap=8
        )
        retrieval_rows.append({
            "variant": "aa_transformer_joint+retrieval",
            "neg_weight": neg_weight,
            "best_alpha": best_alpha,
            "val_mape": float(score_df.iloc[0]["val_mape"]),
            "val_nrmse": float(score_df.iloc[0]["val_nrmse"]),
            "val_uptrend_rate": float(score_df.iloc[0]["val_uptrend_rate"]),
            "holdout_h1_pred": float(holdout["pred"][0]),
            "holdout_h2_pred": float(holdout["pred"][1]),
            "holdout_h1_actual": float(pack["test_eval"]["true_level"][0, 0]),
            "holdout_h2_actual": float(pack["test_eval"]["true_level"][0, 1]),
            "holdout_h1_err_pct": abs(float(holdout["pred"][0] - pack["test_eval"]["true_level"][0, 0])) / float(pack["test_eval"]["true_level"][0, 0]),
            "holdout_h2_err_pct": abs(float(holdout["pred"][1] - pack["test_eval"]["true_level"][0, 1])) / float(pack["test_eval"]["true_level"][0, 1]),
            "holdout_uptrend": bool(holdout["pred"][1] > holdout["pred"][0]),
            "chosen_neighbor_end_idx": holdout["neighbors"][0] if holdout["neighbors"] else -1,
            "neighbor_similarity": holdout["similarities"][0] if holdout["similarities"] else np.nan,
            "base_h1": float(holdout["base_pred"][0]),
            "base_h2": float(holdout["base_pred"][1]),
            "mem_h1": float(holdout["memory_level"][0]) if holdout.get("memory_level") is not None else np.nan,
            "mem_h2": float(holdout["memory_level"][1]) if holdout.get("memory_level") is not None else np.nan,
        })
    df_retrieval = pd.DataFrame(retrieval_rows).sort_values(["neg_weight"]).reset_index(drop=True)
    df_retrieval.to_csv(OUT_RUN / "retrieval_results.csv", index=False)

    best_pack = round1_packs[("aa_transformer_joint", 0.95)]
    gated_rows = []
    cache = precompute_all_windows(best_pack)
    for trigger_q in [0.6, 0.7, 0.8, 0.9]:
        for ratio_floor in [0.7, 0.9]:
            for alpha in [0.25, 0.5]:
                preds = []
                triggered = 0
                for end in best_pack["val_ds"]["end_indices"]:
                    res = retrieve_prediction_for_end(
                        best_pack, cache, int(end),
                        trigger_q=trigger_q, ratio_floor=ratio_floor,
                        key_mode="summary", value_mode="level",
                        top_k=1, alpha=alpha, recency_gap=8
                    )
                    preds.append(res["pred"])
                    triggered += int(res["triggered"])
                preds = np.stack(preds)
                true = best_pack["val_eval"]["true_level"]
                gated_rows.append({
                    "trigger_q": trigger_q,
                    "ratio_floor": ratio_floor,
                    "alpha": alpha,
                    "triggered_val_count": triggered,
                    "val_mape": mape_np(true, preds),
                    "val_nrmse": nrmse_np(true, preds),
                    "val_uptrend_rate": float(np.mean(preds[:, 1] > preds[:, 0])),
                    "selector_score": mape_np(true, preds) + 0.12 * nrmse_np(true, preds) - 0.01 * float(np.mean(preds[:, 1] > preds[:, 0])),
                })
    gated_df = pd.DataFrame(gated_rows).sort_values("selector_score").reset_index(drop=True)
    best_gated = gated_df.iloc[0].to_dict()
    holdout_gated = retrieve_prediction_for_end(
        best_pack, cache, 580,
        trigger_q=float(best_gated["trigger_q"]),
        ratio_floor=float(best_gated["ratio_floor"]),
        key_mode="summary", value_mode="level",
        top_k=1, alpha=float(best_gated["alpha"]), recency_gap=8
    )

    repeat_rows = []
    for seed in [0, 1, 2]:
        _, pack = fit_base_variant("aa_transformer_joint", True, 0.95, 64, seed, "uniform", 0.0, 0.9)
        cache_seed = precompute_all_windows(pack)
        res = retrieve_prediction_for_end(
            pack, cache_seed, 580,
            trigger_q=float(best_gated["trigger_q"]),
            ratio_floor=float(best_gated["ratio_floor"]),
            key_mode="summary", value_mode="level", top_k=1,
            alpha=float(best_gated["alpha"]), recency_gap=8
        )
        repeat_rows.append({
            "seed": seed,
            "base_val_mape": pack["val_eval"]["mape"],
            "base_val_nrmse": pack["val_eval"]["nrmse"],
            "base_holdout_h1": float(pack["test_eval"]["pred_level"][0, 0]),
            "base_holdout_h2": float(pack["test_eval"]["pred_level"][0, 1]),
            "gated_holdout_h1": float(res["pred"][0]),
            "gated_holdout_h2": float(res["pred"][1]),
            "gated_triggered": bool(res["triggered"]),
            "neighbor": res["neighbors"][0] if res["neighbors"] else -1,
            "similarity": res["similarities"][0] if res["similarities"] else np.nan,
        })
    repeat_df = pd.DataFrame(repeat_rows)
    repeat_df.to_csv(OUT_RUN / "aa_transformer_repeatability.csv", index=False)

    probe_end_idx = list(range(64 - 1, 579))
    probe_rows = []
    for end in probe_end_idx:
        future = data[TARGET].values[end + 1 : end + 3]
        anchor = data[TARGET].values[end]
        gain = float(future[-1] - anchor)
        probe_rows.append({
            "end_idx": end,
            "dt_end": str(data.loc[end, "dt"].date()),
            "gain_h2": gain,
            "h1_actual": float(future[0]),
            "h2_actual": float(future[1]),
            "anchor": float(anchor),
        })
    probe_df = pd.DataFrame(probe_rows)
    surge_thr = probe_df["gain_h2"].quantile(0.9)
    surge_df = probe_df[probe_df["gain_h2"] >= surge_thr].copy().reset_index(drop=True)
    surge_end_idx = surge_df["end_idx"].tolist()

    surge_rows = []
    compare_models = {
        "plain_gru": round1_packs[("plain_gru", 1.0)],
        "plain_transformer": round1_packs[("plain_transformer", 1.0)],
        "aa_gru": round1_packs[("aa_gru", 1.0)],
        "aa_transformer_0.95": round1_packs[("aa_transformer_joint", 0.95)],
    }
    for label, pack in compare_models.items():
        surge_ds = make_windows(pack["feat"], surge_end_idx, pack["input_size"])
        ev = evaluate_model(pack["model"], surge_ds)
        pred = ev["pred_level"]
        true = ev["true_level"]
        gain_pred = pred[:, 1] - surge_df["anchor"].to_numpy()
        gain_true = true[:, 1] - surge_df["anchor"].to_numpy()
        surge_rows.append({
            "model": label,
            "surge_mape": mape_np(true, pred),
            "surge_nrmse": nrmse_np(true, pred),
            "surge_uptrend_rate": float(np.mean(pred[:, 1] > pred[:, 0])),
            "surge_gain_mae": float(np.mean(np.abs(gain_pred - gain_true))),
            "surge_h2_mean_pred": float(np.mean(pred[:, 1])),
            "surge_h2_mean_true": float(np.mean(true[:, 1])),
        })
    surge_result_df = pd.DataFrame(surge_rows).sort_values("surge_gain_mae")
    surge_result_df.to_csv(OUT_RUN / "surge_probe_results.csv", index=False)

    combined_holdout = pd.concat([
        df_results.assign(experiment="base_or_aa"),
        df_retrieval.assign(experiment="retrieval_overlay"),
    ], ignore_index=True, sort=False)
    combined_holdout.to_csv(OUT_RUN / "combined_holdout_results.csv", index=False)

    plot_df = pd.DataFrame({
        "model": ["plain_gru", "plain_transformer", "aa_gru", "aa_transformer_0.95", "aa_transformer_0.95+gated_memory"],
        "h1_pred": [
            float(df_results.query("variant=='plain_gru'").iloc[0]["holdout_h1_pred"]),
            float(df_results.query("variant=='plain_transformer'").iloc[0]["holdout_h1_pred"]),
            float(df_results.query("variant=='aa_gru'").iloc[0]["holdout_h1_pred"]),
            float(df_results.query("variant=='aa_transformer_joint' and neg_weight==0.95").iloc[0]["holdout_h1_pred"]),
            float(holdout_gated["pred"][0]),
        ],
        "h2_pred": [
            float(df_results.query("variant=='plain_gru'").iloc[0]["holdout_h2_pred"]),
            float(df_results.query("variant=='plain_transformer'").iloc[0]["holdout_h2_pred"]),
            float(df_results.query("variant=='aa_gru'").iloc[0]["holdout_h2_pred"]),
            float(df_results.query("variant=='aa_transformer_joint' and neg_weight==0.95").iloc[0]["holdout_h2_pred"]),
            float(holdout_gated["pred"][1]),
        ],
    })
    actual_h1 = float(df_results.iloc[0]["holdout_h1_actual"])
    actual_h2 = float(df_results.iloc[0]["holdout_h2_actual"])

    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(plot_df))
    ax.plot(x, plot_df["h1_pred"], marker="o", label="Pred h1")
    ax.plot(x, plot_df["h2_pred"], marker="o", label="Pred h2")
    ax.axhline(actual_h1, linestyle="--", label="Actual h1")
    ax.axhline(actual_h2, linestyle=":", label="Actual h2")
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["model"], rotation=25, ha="right")
    ax.set_ylabel("Brent level")
    ax.set_title("Last holdout 2-step predictions")
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT_RUN / "holdout_prediction_comparison.png", dpi=160)
    plt.close(fig)

    report = f"""
# Brent df-only controlled experiment report

## Scope
This is the round-1 surrogate notebook rerun.

## Data
- target: {TARGET}
- present exogenous variables: {present}
- missing exogenous variables: {missing}

## Best gated retrieval found in this rerun
- trigger_q = {best_gated['trigger_q']}
- ratio_floor = {best_gated['ratio_floor']}
- alpha = {best_gated['alpha']}
- holdout = {holdout_gated['pred'][0]:.3f} / {holdout_gated['pred'][1]:.3f}

## Base comparison
{df_results[['variant','neg_weight','val_mape','val_nrmse','holdout_h1_pred','holdout_h2_pred','holdout_h1_err_pct','holdout_h2_err_pct']].to_string(index=False)}

## Retrieval overlay comparison
{df_retrieval[['neg_weight','best_alpha','val_mape','val_nrmse','holdout_h1_pred','holdout_h2_pred','holdout_h1_err_pct','holdout_h2_err_pct']].to_string(index=False)}

## Repeatability
{repeat_df.to_string(index=False)}
""".strip()
    (OUT_RUN / "experiment_report.md").write_text(report, encoding="utf-8")

print("Round-1 output files:")
for p in sorted(OUT_RUN.glob("*")):
    print(" -", p.name)


## Round 2
input_size, event-weighted training, retrieval rule grid, rare-trigger rule을 재현합니다.

In [ ]:

if FORCE_RECOMPUTE or not all((NOTEBOOK_DIR / p).exists() for p in ["brent_experiment_round2"]):
    print("Running round-2 experiments...")
    OUT_ROUND2.mkdir(exist_ok=True, parents=True)

    base_grid_specs = [
        {"input_size": 64, "neg_weight": 0.95, "weight_scheme": "soft", "weight_strength": 1.0, "threshold_q": 0.9},
        {"input_size": 64, "neg_weight": 0.95, "weight_scheme": "soft", "weight_strength": 0.5, "threshold_q": 0.9},
        {"input_size": 64, "neg_weight": 0.95, "weight_scheme": "binary", "weight_strength": 0.5, "threshold_q": 0.8},
        {"input_size": 64, "neg_weight": 0.95, "weight_scheme": "uniform", "weight_strength": 0.0, "threshold_q": 0.9},
        {"input_size": 64, "neg_weight": 0.90, "weight_scheme": "uniform", "weight_strength": 0.0, "threshold_q": 0.9},
        {"input_size": 64, "neg_weight": 0.95, "weight_scheme": "binary", "weight_strength": 0.5, "threshold_q": 0.9},
        {"input_size": 64, "neg_weight": 0.95, "weight_scheme": "binary", "weight_strength": 1.0, "threshold_q": 0.9},
        {"input_size": 80, "neg_weight": 0.95, "weight_scheme": "uniform", "weight_strength": 0.0, "threshold_q": 0.9},
    ]

    base_rows = []
    base_packs = {}
    for spec in base_grid_specs:
        row, pack = fit_base_variant(
            "aa_transformer_joint",
            use_aa=True,
            neg_weight=spec["neg_weight"],
            input_size=spec["input_size"],
            seed=0,
            weight_scheme=spec["weight_scheme"],
            weight_strength=spec["weight_strength"],
            threshold_q=spec["threshold_q"],
        )
        base_rows.append({
            "input_size": spec["input_size"],
            "neg_weight": spec["neg_weight"],
            "weight_scheme": spec["weight_scheme"],
            "weight_strength": spec["weight_strength"],
            "threshold_q": spec["threshold_q"],
            "val_mape": row["val_mape"],
            "val_nrmse": row["val_nrmse"],
            "val_uptrend": row["val_uptrend_rate"],
            "holdout_h1": row["holdout_h1_pred"],
            "holdout_h2": row["holdout_h2_pred"],
            "holdout_uptrend": row["holdout_uptrend"],
            "h1_err_pct": row["holdout_h1_err_pct"],
            "h2_err_pct": row["holdout_h2_err_pct"],
        })
        key = (spec["input_size"], spec["neg_weight"], spec["weight_scheme"], spec["weight_strength"], spec["threshold_q"])
        base_packs[key] = pack

    round2_base_df = pd.DataFrame(base_rows).sort_values(["val_mape", "val_nrmse"]).reset_index(drop=True)
    round2_base_df.to_csv(OUT_ROUND2 / "round2_base_grid.csv", index=False)

    candidate_specs = {
        "best_val64": (64, 0.95, "soft", 1.0, 0.9),
        "base80": (80, 0.95, "uniform", 0.0, 0.9),
    }

    def run_retrieval_grid(candidate_name, pack, trigger_q_values, ratio_floor_values, key_modes, value_modes, top_k_values, alpha_values):
        cache = precompute_all_windows(pack)
        rows = []
        for trigger_q in trigger_q_values:
            for ratio_floor in ratio_floor_values:
                for key_mode in key_modes:
                    beta_values = [None] if key_mode == "summary" else [0.5, 1.0]
                    for beta in beta_values:
                        for value_mode in value_modes:
                            for top_k in top_k_values:
                                for alpha in alpha_values:
                                    preds = []
                                    triggered = 0
                                    for end in pack["val_ds"]["end_indices"]:
                                        res = retrieve_prediction_for_end(
                                            pack, cache, int(end),
                                            trigger_q=trigger_q,
                                            ratio_floor=ratio_floor,
                                            key_mode=key_mode,
                                            beta=beta,
                                            value_mode=value_mode,
                                            top_k=top_k,
                                            alpha=alpha,
                                            recency_gap=8,
                                        )
                                        preds.append(res["pred"])
                                        triggered += int(res["triggered"])
                                    preds = np.stack(preds)
                                    true = pack["val_eval"]["true_level"]
                                    holdout = retrieve_prediction_for_end(
                                        pack, cache, 580,
                                        trigger_q=trigger_q, ratio_floor=ratio_floor,
                                        key_mode=key_mode, beta=beta,
                                        value_mode=value_mode, top_k=top_k, alpha=alpha, recency_gap=8
                                    )
                                    row = {
                                        "trigger_q": trigger_q,
                                        "ratio_floor": ratio_floor,
                                        "key_mode": key_mode,
                                        "beta": beta,
                                        "value_mode": value_mode,
                                        "top_k": top_k,
                                        "alpha": alpha,
                                        "triggered_val": triggered,
                                        "val_mape": mape_np(true, preds),
                                        "val_nrmse": nrmse_np(true, preds),
                                        "val_uptrend": float(np.mean(preds[:, 1] > preds[:, 0])),
                                    }
                                    row["selector_score"] = row["val_mape"] + 0.12 * row["val_nrmse"] - 0.01 * row["val_uptrend"]
                                    row["holdout_h1"] = float(holdout["pred"][0])
                                    row["holdout_h2"] = float(holdout["pred"][1])
                                    row["holdout_uptrend"] = bool(holdout["pred"][1] > holdout["pred"][0])
                                    row["holdout_h1_err_pct"] = abs(float(holdout["pred"][0] - pack["test_eval"]["true_level"][0, 0])) / float(pack["test_eval"]["true_level"][0, 0])
                                    row["holdout_h2_err_pct"] = abs(float(holdout["pred"][1] - pack["test_eval"]["true_level"][0, 1])) / float(pack["test_eval"]["true_level"][0, 1])
                                    row["pass_fallback"] = bool((holdout["pred"][0] >= 75) and (holdout["pred"][1] >= 80))
                                    row["pass_15pct"] = bool((row["holdout_h1_err_pct"] <= 0.15) and (row["holdout_h2_err_pct"] <= 0.15))
                                    row["chosen_neighbors"] = str(holdout["neighbors"])
                                    row["candidate"] = candidate_name
                                    rows.append(row)
        return pd.DataFrame(rows)

    retrieval_frames = []
    highq_frames = []
    for candidate_name, key in candidate_specs.items():
        pack = base_packs[key]
        retrieval_frames.append(run_retrieval_grid(
            candidate_name, pack,
            trigger_q_values=[0.85, 0.90],
            ratio_floor_values=[0.8, 0.9],
            key_modes=["summary", "summary_event"],
            value_modes=["level", "delta", "return"],
            top_k_values=[1, 2],
            alpha_values=[0.25, 0.30, 0.35],
        ))
        highq_frames.append(run_retrieval_grid(
            candidate_name, pack,
            trigger_q_values=[0.95, 0.97, 0.99],
            ratio_floor_values=[0.8, 0.9],
            key_modes=["summary", "summary_event"],
            value_modes=["level"],
            top_k_values=[1, 2],
            alpha_values=[0.25, 0.30, 0.35, 0.40],
        ))

    round2_retrieval_df = pd.concat(retrieval_frames, ignore_index=True)
    round2_retrieval_df.to_csv(OUT_ROUND2 / "round2_retrieval_grid.csv", index=False)

    round2_highq_df = pd.concat(highq_frames, ignore_index=True)
    round2_highq_df.to_csv(OUT_ROUND2 / "round2_highq_retrieval_grid.csv", index=False)

    selected_rule = {
        "trigger_q": 0.95,
        "ratio_floor": 0.8,
        "key_mode": "summary",
        "beta": None,
        "value_mode": "level",
        "top_k": 1,
        "alpha": 0.40,
        "candidate": "best_val64",
    }

    selected_pack = base_packs[candidate_specs["best_val64"]]
    selected_cache = precompute_all_windows(selected_pack)
    stability_rows = []
    for seed in [0, 1, 2]:
        _, pack_seed = fit_base_variant(
            "aa_transformer_joint", use_aa=True, neg_weight=0.95, input_size=64,
            seed=seed, weight_scheme="soft", weight_strength=1.0, threshold_q=0.9
        )
        cache_seed = precompute_all_windows(pack_seed)
        res = retrieve_prediction_for_end(
            pack_seed, cache_seed, 580,
            trigger_q=selected_rule["trigger_q"],
            ratio_floor=selected_rule["ratio_floor"],
            key_mode=selected_rule["key_mode"],
            beta=selected_rule["beta"],
            value_mode=selected_rule["value_mode"],
            top_k=selected_rule["top_k"],
            alpha=selected_rule["alpha"],
            recency_gap=8,
        )
        stability_rows.append({
            "seed": seed,
            "base_h1": float(pack_seed["test_eval"]["pred_level"][0, 0]),
            "base_h2": float(pack_seed["test_eval"]["pred_level"][0, 1]),
            "final_h1": float(res["pred"][0]),
            "final_h2": float(res["pred"][1]),
            "h1_err_pct": abs(float(res["pred"][0] - pack_seed["test_eval"]["true_level"][0, 0])) / float(pack_seed["test_eval"]["true_level"][0, 0]),
            "h2_err_pct": abs(float(res["pred"][1] - pack_seed["test_eval"]["true_level"][0, 1])) / float(pack_seed["test_eval"]["true_level"][0, 1]),
            "uptrend": bool(res["pred"][1] > res["pred"][0]),
            "triggered": bool(res["triggered"]),
            "neighbors": str(res["neighbors"]),
            "trigger_thr": float(res["trigger_threshold"]),
            "qscore": float(res["query_score"]),
        })
    stability_df = pd.DataFrame(stability_rows)
    stability_df.to_csv(OUT_ROUND2 / "round2_selected_rule_seed_stability.csv", index=False)

    probe_df, probe_summary = historical_probe_rule(
        selected_pack,
        rule={"trigger_q": 0.95, "ratio_floor": 0.8, "top_k": 1, "alpha": 0.40},
        key_mode="summary",
        beta=None,
        value_mode="level",
        recency_gap=8,
    )
    probe_df.to_csv(OUT_ROUND2 / "round2_selected_rule_historical_probe.csv", index=False)

    pass15_rules = pd.concat([round2_retrieval_df, round2_highq_df], ignore_index=True)
    pass15_rules = pass15_rules[pass15_rules["pass_15pct"]].copy().reset_index(drop=True)

    tradeoff_rows = []
    for _, r in pass15_rules.iterrows():
        candidate_key = candidate_specs[r["candidate"]]
        pack = base_packs[candidate_key]
        probe_tmp, summary_tmp = historical_probe_rule(
            pack,
            rule={"trigger_q": float(r["trigger_q"]), "ratio_floor": float(r["ratio_floor"]), "top_k": int(r["top_k"]), "alpha": float(r["alpha"])},
            key_mode=r["key_mode"],
            beta=(None if pd.isna(r["beta"]) else float(r["beta"])),
            value_mode=r["value_mode"],
            recency_gap=8,
        )
        tradeoff_rows.append({
            "candidate": r["candidate"],
            "trigger_q": r["trigger_q"],
            "ratio_floor": r["ratio_floor"],
            "key_mode": r["key_mode"],
            "beta": r["beta"],
            "value_mode": r["value_mode"],
            "top_k": r["top_k"],
            "alpha": r["alpha"],
            "holdout_h1": r["holdout_h1"],
            "holdout_h2": r["holdout_h2"],
            "holdout_h1_err_pct": r["holdout_h1_err_pct"],
            "holdout_h2_err_pct": r["holdout_h2_err_pct"],
            "n_triggered": summary_tmp["n_triggered"],
            "trigger_rate": summary_tmp["trigger_rate"],
            "triggered_improved_h2_rate": summary_tmp["triggered_improved_h2_rate"],
            "triggered_base_h2_err_mean": summary_tmp["triggered_base_h2_err_mean"],
            "triggered_rule_h2_err_mean": summary_tmp["triggered_rule_h2_err_mean"],
            "surge_triggered_count": summary_tmp["surge_triggered_count"],
            "surge_base_h2_err_mean": summary_tmp["surge_base_h2_err_mean"],
            "surge_rule_h2_err_mean": summary_tmp["surge_rule_h2_err_mean"],
        })
    tradeoff_df = pd.DataFrame(tradeoff_rows).sort_values(["holdout_h2_err_pct", "trigger_rate"]).reset_index(drop=True)
    tradeoff_df.to_csv(OUT_ROUND2 / "round2_pass15_historical_tradeoff.csv", index=False)

    candidate_rows = []
    for label, spec in candidate_specs.items():
        pack = base_packs[spec]
        candidate_rows.append({
            "candidate": label,
            "base_h1": float(pack["test_eval"]["pred_level"][0, 0]),
            "base_h2": float(pack["test_eval"]["pred_level"][0, 1]),
        })
    candidate_df = pd.DataFrame(candidate_rows)

    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(4)
    rule_labels = ["base64", "rare_q95", "pass_q90", "base80"]
    rare_holdout = retrieve_prediction_for_end(
        selected_pack, selected_cache, 580, trigger_q=0.95, ratio_floor=0.8,
        key_mode="summary", beta=None, value_mode="level", top_k=1, alpha=0.40, recency_gap=8
    )
    pass_holdout = retrieve_prediction_for_end(
        selected_pack, selected_cache, 580, trigger_q=0.90, ratio_floor=0.8,
        key_mode="summary", beta=None, value_mode="level", top_k=1, alpha=0.30, recency_gap=8
    )
    h1_vals = [
        float(selected_pack["test_eval"]["pred_level"][0, 0]),
        float(rare_holdout["pred"][0]),
        float(pass_holdout["pred"][0]),
        float(base_packs[candidate_specs["base80"]]["test_eval"]["pred_level"][0, 0]),
    ]
    h2_vals = [
        float(selected_pack["test_eval"]["pred_level"][0, 1]),
        float(rare_holdout["pred"][1]),
        float(pass_holdout["pred"][1]),
        float(base_packs[candidate_specs["base80"]]["test_eval"]["pred_level"][0, 1]),
    ]
    actual_h1 = float(selected_pack["test_eval"]["true_level"][0, 0])
    actual_h2 = float(selected_pack["test_eval"]["true_level"][0, 1])

    ax.plot(x, h1_vals, marker="o", label="Pred h1")
    ax.plot(x, h2_vals, marker="o", label="Pred h2")
    ax.axhline(actual_h1, linestyle="--", label="Actual h1")
    ax.axhline(actual_h2, linestyle=":", label="Actual h2")
    ax.set_xticks(x)
    ax.set_xticklabels(rule_labels)
    ax.set_ylabel("Brent level")
    ax.set_title("Round-2 holdout comparison")
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT_ROUND2 / "round2_holdout_comparison.png", dpi=160)
    plt.close(fig)

    report = f"""
# Brent round-2 df-only experiment report

## Scope
- Continued from the previous df-only surrogate experiments.
- Target: {TARGET}
- Horizon: {H}
- Base split preserved with final holdout anchored at 2026-02-23 -> predicts 2026-03-02 and 2026-03-09.

## Main findings
- Base AA-Transformer remained trapped near 72-73 on the last holdout in this rerun as well.
- The strongest retrieval gain on the final holdout came from `value_mode=level`.
- Selected rare-trigger rule:
  {json.dumps(selected_rule, ensure_ascii=False)}

## Rare-trigger seed stability
{stability_df.to_string(index=False)}

## Historical probe summary
{json.dumps(probe_summary, ensure_ascii=False, indent=2)}
""".strip()
    (OUT_ROUND2 / "round2_report.md").write_text(report, encoding="utf-8")

print("Round-2 output files:")
for p in sorted(OUT_ROUND2.glob("*")):
    print(" -", p.name)


## Compliant Round
금지사항을 지키는 준수형 surrogate 실험을 재현합니다.

In [ ]:

def compliant_latent_memory_adjust(pack, trigger_q=0.9, alpha=0.25, recency_gap=8):
    cache = precompute_all_windows(pack)
    event_map = pack["event_map"]
    train_scores = np.array([event_map[e] for e in range(pack["input_size"] - 1, TRAIN_END_MAX + 1)], dtype=np.float32)
    trigger_thr = float(np.quantile(train_scores, trigger_q))

    def adjust_for_end(query_end):
        base_pred = cache["pred_by_end"][int(query_end)]
        base_summary = cache["summary_by_end"][int(query_end)]
        qscore = event_map[int(query_end)]
        if qscore < trigger_thr:
            return {
                "pred": base_pred.copy(),
                "triggered": False,
                "similarity": 0.0,
                "trigger_threshold": trigger_thr,
                "query_score": qscore,
            }
        candidates = [e for e in cache["all_end_idx"] if e <= int(query_end) - recency_gap and e <= TRAIN_END_MAX]
        keys = np.stack([cache["summary_by_end"][e] for e in candidates])
        q = base_summary / (np.linalg.norm(base_summary) + 1e-8)
        keys = keys / (np.linalg.norm(keys, axis=1, keepdims=True) + 1e-8)
        sims = keys @ q
        idx = int(np.argmax(sims))
        mem_summary = cache["summary_by_end"][candidates[idx]]
        blended_summary = (1 - alpha) * base_summary + alpha * mem_summary
        pred_diff = pack["model"].decode_summary(torch.tensor(blended_summary[None, :], dtype=torch.float32)).detach().numpy()[0]
        pred_level = np.array([cache["anchor_by_end"][int(query_end)] + pred_diff[0], cache["anchor_by_end"][int(query_end)] + pred_diff[0] + pred_diff[1]], dtype=np.float32)
        return {
            "pred": pred_level,
            "triggered": True,
            "similarity": float(sims[idx]),
            "trigger_threshold": trigger_thr,
            "query_score": qscore,
        }

    val_preds = []
    val_triggered = 0
    for end in pack["val_ds"]["end_indices"]:
        res = adjust_for_end(int(end))
        val_preds.append(res["pred"])
        val_triggered += int(res["triggered"])
    val_preds = np.stack(val_preds)

    holdout = adjust_for_end(580)
    true = pack["val_eval"]["true_level"]
    return {
        "val_mape": mape_np(true, val_preds),
        "val_nrmse": nrmse_np(true, val_preds),
        "val_uptrend": float(np.mean(val_preds[:, 1] > val_preds[:, 0])),
        "val_trigger_rate": float(val_triggered / len(pack["val_ds"]["end_indices"])),
        "holdout": holdout,
    }

if FORCE_RECOMPUTE or not all((NOTEBOOK_DIR / p).exists() for p in ["compliant_experiment_round"]):
    print("Running compliant-only experiments...")
    OUT_COMPLIANT.mkdir(exist_ok=True, parents=True)

    rows = []

    # plain_gru
    row, pack = fit_base_variant("plain_gru", use_aa=False, neg_weight=1.0, input_size=64, seed=0, weight_scheme="uniform", weight_strength=0.0, threshold_q=0.9)
    rows.append({
        "variant": "plain_gru", "input_size": 64, "neg_scale": 1.0, "trigger_q": np.nan,
        "val_mape": row["val_mape"], "val_nrmse": row["val_nrmse"], "val_uptrend": row["val_uptrend_rate"],
        "h1_pred": row["holdout_h1_pred"], "h2_pred": row["holdout_h2_pred"],
        "h1_actual": row["holdout_h1_actual"], "h2_actual": row["holdout_h2_actual"],
        "h1_err_pct": row["holdout_h1_err_pct"], "h2_err_pct": row["holdout_h2_err_pct"],
        "uptrend_pass": row["holdout_uptrend"],
        "fallback_pass": bool((row["holdout_h1_pred"] >= 75) and (row["holdout_h2_pred"] >= 80)),
        "strict15_pass": bool((row["holdout_h1_err_pct"] <= 0.15) and (row["holdout_h2_err_pct"] <= 0.15)),
        "test_mem_triggered": False, "test_mem_similarity": 0.0, "trigger_threshold": np.nan,
        "seed": 0, "val_trigger_rate": 0.0,
    })

    aa_specs = [
        ("aa_gru", 64, 1.0, None),
        ("aa_gru", 64, 0.95, None),
        ("aa_gru", 80, 0.95, None),
        ("aa_gru_memory", 64, 0.95, 0.90),
        ("aa_gru_memory", 64, 0.95, 0.95),
        ("aa_gru_memory", 80, 0.95, 0.90),
        ("aa_gru_memory", 80, 0.95, 0.95),
    ]

    for variant, input_size, neg_scale, trigger_q in aa_specs:
        row, pack = fit_base_variant("aa_gru", use_aa=True, neg_weight=neg_scale, input_size=input_size, seed=0, weight_scheme="uniform", weight_strength=0.0, threshold_q=0.9)
        if variant == "aa_gru":
            rows.append({
                "variant": variant, "input_size": input_size, "neg_scale": neg_scale, "trigger_q": np.nan,
                "val_mape": row["val_mape"], "val_nrmse": row["val_nrmse"], "val_uptrend": row["val_uptrend_rate"],
                "h1_pred": row["holdout_h1_pred"], "h2_pred": row["holdout_h2_pred"],
                "h1_actual": row["holdout_h1_actual"], "h2_actual": row["holdout_h2_actual"],
                "h1_err_pct": row["holdout_h1_err_pct"], "h2_err_pct": row["holdout_h2_err_pct"],
                "uptrend_pass": row["holdout_uptrend"],
                "fallback_pass": bool((row["holdout_h1_pred"] >= 75) and (row["holdout_h2_pred"] >= 80)),
                "strict15_pass": bool((row["holdout_h1_err_pct"] <= 0.15) and (row["holdout_h2_err_pct"] <= 0.15)),
                "test_mem_triggered": False, "test_mem_similarity": 0.0, "trigger_threshold": np.nan,
                "seed": 0, "val_trigger_rate": 0.0,
            })
        else:
            comp = compliant_latent_memory_adjust(pack, trigger_q=trigger_q, alpha=0.25, recency_gap=8)
            h1_pred = float(comp["holdout"]["pred"][0]); h2_pred = float(comp["holdout"]["pred"][1])
            h1_actual = float(row["holdout_h1_actual"]); h2_actual = float(row["holdout_h2_actual"])
            rows.append({
                "variant": variant, "input_size": input_size, "neg_scale": neg_scale, "trigger_q": trigger_q,
                "val_mape": comp["val_mape"], "val_nrmse": comp["val_nrmse"], "val_uptrend": comp["val_uptrend"],
                "h1_pred": h1_pred, "h2_pred": h2_pred,
                "h1_actual": h1_actual, "h2_actual": h2_actual,
                "h1_err_pct": abs(h1_pred - h1_actual) / h1_actual,
                "h2_err_pct": abs(h2_pred - h2_actual) / h2_actual,
                "uptrend_pass": bool(h2_pred > h1_pred),
                "fallback_pass": bool((h1_pred >= 75) and (h2_pred >= 80)),
                "strict15_pass": bool((abs(h1_pred - h1_actual) / h1_actual <= 0.15) and (abs(h2_pred - h2_actual) / h2_actual <= 0.15)),
                "test_mem_triggered": bool(comp["holdout"]["triggered"]),
                "test_mem_similarity": float(comp["holdout"]["similarity"]),
                "trigger_threshold": float(comp["holdout"]["trigger_threshold"]),
                "seed": 0,
                "val_trigger_rate": comp["val_trigger_rate"],
            })

    compliant_df = pd.DataFrame(rows).sort_values(["variant", "input_size", "neg_scale", "trigger_q"], na_position="last").reset_index(drop=True)
    compliant_df.to_csv(OUT_COMPLIANT / "compliant_surrogate_results.csv", index=False)

    best_idx = compliant_df["h2_pred"].idxmax()
    best_row = compliant_df.loc[best_idx].to_dict()

    report = f"""
# Compliant-only experiment round

## Scope
This round intentionally excluded any mechanism that would violate the user's restrictions:
- no horizon-specific continuation bonus
- no explicit positive drift/uplift
- no use of recent target rise to boost future horizons
- no historical future-level posthoc blend
- no decoder-internal cumsum / monotonic forcing branch in the tested surrogate models

## Data
- target: {TARGET}
- allowed variables requested: {ALLOWED}
- present in uploaded df.csv: {present}
- missing in uploaded df.csv: {missing}
- final holdout actual: h1={float(actual_holdout[TARGET].iloc[0]):.4f}, h2={float(actual_holdout[TARGET].iloc[1]):.4f}

## Self-run compliant surrogate results
{compliant_df.to_string(index=False)}

## Takeaways
- All self-run compliant surrogate variants preserved the safety constraints.
- None of these simplified compliant surrogates are guaranteed to beat the repo keep family.
- Best row in this rerun:
  {best_row}
""".strip()
    (OUT_COMPLIANT / "compliant_surrogate_report.md").write_text(report, encoding="utf-8")

    summary_md = f"""
# Current main + self-run compliant-only summary

This notebook only reruns the local df-only compliant surrogate family.

## My self-run compliant-only experiments
{compliant_df[['variant','input_size','neg_scale','trigger_q','h1_pred','h2_pred','h1_err_pct','h2_err_pct','uptrend_pass','fallback_pass','strict15_pass']].to_string(index=False)}
""".strip()
    (OUT_COMPLIANT / "current_main_and_selfrun_summary.md").write_text(summary_md, encoding="utf-8")

print("Compliant output files:")
for p in sorted(OUT_COMPLIANT.glob("*")):
    print(" -", p.name)


## Summary

In [ ]:

print("\n==== SUMMARY TABLES ====")

for label, folder in [("round1", OUT_RUN), ("round2", OUT_ROUND2), ("compliant", OUT_COMPLIANT)]:
    print(f"\n[{label}]")
    if folder.exists():
        for p in sorted(folder.glob("*")):
            print(" -", p.name)

def maybe_read_csv(path):
    return pd.read_csv(path) if path.exists() else None

run_df = maybe_read_csv(OUT_RUN / "combined_holdout_results.csv")
round2_df = maybe_read_csv(OUT_ROUND2 / "round2_highq_retrieval_grid.csv")
comp_df = maybe_read_csv(OUT_COMPLIANT / "compliant_surrogate_results.csv")

if run_df is not None:
    print("\nRound-1 holdout preview:")
    display(run_df.head())

if round2_df is not None:
    print("\nRound-2 top high-q rows:")
    display(round2_df.sort_values(["holdout_h2_err_pct", "holdout_h1_err_pct"]).head(10))

if comp_df is not None:
    print("\nCompliant preview:")
    display(comp_df)

if SAVE_EXECUTION_BUNDLE:
    bundle_path = NOTEBOOK_DIR / "brent_full_experiment_reproduction_bundle.zip"
    with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as zf:
        nb_name = "brent_full_experiment_reproduction.ipynb"
        if (NOTEBOOK_DIR / nb_name).exists():
            zf.write(NOTEBOOK_DIR / nb_name, arcname=nb_name)
        for folder in [OUT_RUN, OUT_ROUND2, OUT_COMPLIANT]:
            if folder.exists():
                for p in folder.rglob("*"):
                    if p.is_file():
                        zf.write(p, arcname=str(p.relative_to(NOTEBOOK_DIR)))
        if SNAPSHOT_ROOT is not None and SNAPSHOT_ROOT.exists():
            for p in SNAPSHOT_ROOT.rglob("*"):
                if p.is_file():
                    zf.write(p, arcname=str(Path("brent_full_experiment_snapshots") / p.relative_to(SNAPSHOT_ROOT)))
    print("\nBundle saved to:", bundle_path)
